# Practical Statistics for Data Scientists (Python)
# Chapter 5. Classification
> (c) 2019 Peter C. Bruce, Andrew Bruce, Peter Gedeck

**Reproduksi dan pembahasan:** Nanda Pratama ITM · NIM 101032330212 · TK 47 01.

Kode diadaptasi dari repositori resmi Peter Bruce, Andrew Bruce, dan Peter Gedeck (GPL-3.0); bukan klaim kode orisinal mahasiswa. Penjelasan Indonesia dan eksperimen tambahan disusun dengan bantuan AI. Sumber serta koreksi tercatat pada README dan docs/PERUBAHAN.md.

## Petunjuk penggunaan dan sumber

Baca teori berikut, kemudian jalankan sel kode berurutan dari atas ke bawah. Bagian **Reproduksi kode buku** mempertahankan semua sel Python nonkosong sumber resmi dengan koreksi yang dicatat. Bagian **Eksperimen tambahan** adalah pengembangan edukatif. Notebook gabungan ini telah dijalankan berurutan pada kernel baru, termasuk eksperimen tambahan.

Referensi utama: Bruce, Bruce, & Gedeck (2020), *Practical Statistics for Data Scientists*, edisi ke-2. [Kode resmi](https://github.com/gedeck/practical-statistics-for-data-scientists), commit `8a6d3bb6468e979c861d4b37215e1413702dfdfa`. Lihat [catatan perubahan](../docs/PERUBAHAN.md).

# Ringkasan dan pendalaman teori Bab 5

Klasifikasi memprediksi kelas atau peluang kelas. Data pinjaman dipakai untuk Naive Bayes, LDA, regresi logistik, evaluasi, dan strategi ketidakseimbangan kelas. Definisi positif harus eksplisit; pada eksperimen tambahan kelas positif adalah default.


## Naive Bayes
Bayes memberi $P(C|x)\propto P(C)P(x|C)$. Distribusi bersama semua fitur membutuhkan sangat banyak data. Naive Bayes menyederhanakan melalui independensi bersyarat: $P(x|C)=\prod_jP(x_j|C)$. Asumsi ini sering tidak sepenuhnya benar, namun klasifikasi dapat tetap berguna; probabilitas dapat terlalu percaya diri jika fitur saling bergantung.

Sumber menggunakan MultinomialNB pada indikator kategori. GaussianNB mendukung numerik kontinu dengan asumsi Gaussian per kelas, sehingga pernyataan lama bahwa scikit-learn tidak mendukung numerik untuk Naive Bayes tidak berlaku umum. Smoothing alpha mencegah frekuensi nol mematikan likelihood. `fit_prior=False` memakai prior seragam, berbeda dari prior empiris. One-hot multinomial juga berbeda dari categorical Naive Bayes langsung.


## Discriminant analysis
LDA memodelkan fitur per kelas dengan normal multivariat dan kovarians bersama. Matriks kovarians memuat varians fitur dan pergerakan bersamanya. Kovarians bersama menghasilkan batas keputusan linear pada log-rasio posterior. Arah diskriminan memisahkan mean kelas relatif terhadap variasi dalam kelas. QDA mengizinkan kovarians berbeda sehingga batasnya kuadratik, dengan lebih banyak parameter. Kovarians singular dan penyimpangan distribusi berat dapat mengganggu LDA.


## Logistik dan GLM
Regresi logistik memakai $\log[p/(1-p)]=\beta_0+\sum_j\beta_jx_j$ dan sigmoid $p=1/(1+e^{-\eta})$. Respons biner dimodelkan Bernoulli/binomial; parameter diestimasi melalui likelihood, bukan least squares biasa. GLM menghubungkan keluarga distribusi respons, linear predictor, dan link. Regresi linear memakai identity link pada Gaussian; logistik memakai logit pada binomial.

Koefisien b adalah perubahan log-odds per satuan fitur, dengan fitur lain tetap. Odds ratio $e^b$ bukan kenaikan probabilitas sebesar b atau e^b; perubahan peluang bergantung baseline. Urutan `classes_` menentukan kolom `predict_proba`. Memilih kolom 1 tanpa memeriksa label dapat membalik kesimpulan. C sangat besar pada reproduksi mendekati fit tanpa penalti. Separation dapat membuat koefisien tak berhingga atau tidak stabil.


## Penilaian model
Deviance membandingkan kecocokan dengan model jenuh. AIC menambahkan penalti kompleksitas tetapi tidak menggantikan test set. Linearitas logistik berlaku pada logit, bukan pada probabilitas. Spline memperluas bentuk hubungan. Residual/diagnostik membantu mendeteksi spesifikasi salah dan pengamatan berpengaruh.

Beberapa metrik sumber dihitung pada training dan merupakan apparent performance. Tambahan notebook memakai split terstratifikasi, dengan fitting hanya pada training. Borrower_score yang disediakan sumber tidak digunakan pada evaluasi tambahan karena dapat dibentuk menggunakan label dari luar training. Untuk penggunaan nyata, validasi temporal kredit sering lebih relevan daripada split acak.


## Confusion matrix dan kelas langka
Dengan default sebagai positif, TP adalah default terdeteksi, FN default terlewat, FP pinjaman lancar ditandai default, dan TN lancar yang benar. Accuracy $(TP+TN)/N$, precision $TP/(TP+FP)$, recall $TP/(TP+FN)$, specificity $TN/(TN+FP)$. F1 adalah rata-rata harmonik precision dan recall. Jika denominator nol, kebijakan perhitungan harus jelas.

Pada kelas langka, menebak semua negatif dapat memberi accuracy tinggi dengan recall nol. Precision mengukur keandalan alarm; recall mengukur cakupan deteksi positif. Threshold mengubah kompromi keduanya. Threshold 0.5 bukan keharusan; pilih dengan validation set dan biaya keputusan, lalu evaluasi sekali di test.


## ROC, AUC, precision-recall, dan lift
ROC membandingkan true-positive rate dengan false-positive rate pada berbagai threshold. AUC mengukur kemampuan ranking: peluang skor positif lebih tinggi daripada negatif, dengan penanganan ties. AUC 0.5 setara ranking acak, bukan berarti accuracy pasti 50%. AUC tidak menilai kalibrasi dan dapat tampak baik pada kelas sangat jarang meski precision rendah.

Precision-recall curve dan average precision berfokus pada kelas positif. Lift pada fraksi skor teratas membandingkan proporsi positif kelompok itu dengan prevalensi keseluruhan. Pilih metrik sesuai kapasitas tindak lanjut dan biaya, bukan setelah melihat angka yang paling bagus.


## Imbalance dan biaya
Undersampling mengurangi mayoritas dan dapat membuang informasi. Oversampling mereplikasi minoritas. Class weight mengubah kontribusi loss. SMOTE menginterpolasi tetangga minoritas; ADASYN lebih berfokus area yang sulit. Interpolasi dummy kategori bisa menghasilkan nilai yang tidak bermakna; gunakan metode sesuai tipe fitur, misalnya SMOTENC jika diperlukan.

Resampling hanya dilakukan pada training dan di dalam fold CV. Resampling sebelum split menyebabkan kebocoran. Perubahan prior efektif dapat membuat probabilitas perlu kalibrasi/koreksi untuk deployment. Cost-based classification mempertimbangkan kerugian FN dan FP, sehingga accuracy tertinggi belum tentu memberi biaya terendah.


## Eksplorasi prediksi dan kesimpulan
Batas keputusan memperlihatkan fleksibilitas model, bukan bukti model terumit paling baik. LinearGAM pada target 0/1 dalam sumber adalah demonstrasi permukaan skor, bukan probabilitas yang dijamin pada [0,1]; logistic GAM lebih sesuai untuk peluang biner. Interpretasikan confusion matrix, ROC-AUC, dan average precision bersama prevalensi dan baseline. Konsep loss, imbalance, kelas positif, serta threshold juga berlaku pada neural network classifier.


# Reproduksi kode buku
Sel berikut diadaptasi dari penulis asli. Komentar Inggris dipertahankan untuk memudahkan pencocokan sumber. Teori serta interpretasi Indonesia tersedia di atas dan pada bagian akhir.

## Persiapan kernel: jalankan sebelum kode bab

`ModuleNotFoundError` berarti modul belum tersedia di kernel yang dipilih. Sel berikut memeriksa dependensi dan memasang paket yang diperlukan **ke Python yang sedang dipakai notebook**, bukan ke Python lain. Pemasangan pertama memerlukan internet dan dapat memakan beberapa menit. Gunakan Python 3.12 (versi pengujian). Setelah itu, jalankan kode bab dari atas ke bawah. Jika masih gagal, perhatikan nama modul dan pesan pemasangan yang muncul; jangan melewati sel persiapan.

In [ ]:
# Jalankan sel ini lebih dahulu: paket dipasang ke Python/kernel yang aktif.
import sys
import importlib.util
import importlib
import subprocess
from pathlib import Path

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'requirements-lock.txt').is_file() and (p / 'data').is_dir()), None)
if REPO is None:
    raise FileNotFoundError('Buka folder Practical-Statistics-for-Data-Scientists di VS Code, lalu buka notebook dari folder notebooks.')

paket = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'sklearn': 'scikit-learn',
    'statsmodels': 'statsmodels', 'wquantiles': 'wquantiles', 'pygam': 'pygam',
    'dmba': 'dmba', 'pydotplus': 'pydotplus', 'imblearn': 'imbalanced-learn',
    'xgboost': 'xgboost', 'prince': 'prince', 'adjustText': 'adjustText',
    'IPython': 'ipython', 'graphviz': 'graphviz'
}
kurang = [nama for modul, nama in paket.items() if importlib.util.find_spec(modul) is None]
print('Python/kernel aktif:', sys.executable)
print('Versi Python:', sys.version.split()[0])
if kurang:
    print('Paket belum tersedia:', ', '.join(kurang))
    print('Memasang dependensi ke kernel ini. Internet diperlukan pada pemasangan pertama.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-lock.txt')])
    importlib.invalidate_caches()
    print('Pemasangan selesai. Jika library sudah diimpor sebelum pemasangan, pilih Restart Kernel lalu Run All.')
else:
    print('Semua dependensi tersedia. Lanjutkan sel berikutnya secara berurutan.')

In [ ]:
import random
import numpy as np
random.seed(42)
np.random.seed(42)
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

Import required Python packages.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.naive_bayes import MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression #, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score

import statsmodels.api as sm

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from pygam import LinearGAM, s, f, l


from dmba import classificationSummary

import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
DATA = next((p / 'data' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'state.csv').exists()), None)
assert DATA is not None, 'Jalankan notebook dari folder repositori atau notebooks.'
print('Data:', DATA)

Define paths to data sets. If you don't keep your data in the same directory as the code, adapt the path names.

In [ ]:
LOAN3000_CSV = DATA / 'loan3000.csv'
LOAN_DATA_CSV = DATA / 'loan_data.csv.gz'
FULL_TRAIN_SET_CSV = DATA / 'full_train_set.csv.gz'

# Naive Bayes
## The Naive Solution

In [ ]:
loan_data = pd.read_csv(LOAN_DATA_CSV)

# convert to categorical
loan_data.outcome = loan_data.outcome.astype('category')
loan_data.outcome.cat.reorder_categories(['paid off', 'default'])
loan_data.purpose_ = loan_data.purpose_.astype('category')
loan_data.home_ = loan_data.home_.astype('category')
loan_data.emp_len_ = loan_data.emp_len_.astype('category')

predictors = ['purpose_', 'home_', 'emp_len_']
outcome = 'outcome'
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='', dtype=int)
y = loan_data[outcome]

naive_model = MultinomialNB(alpha=0.01, fit_prior=True)
naive_model = MultinomialNB(alpha=1e-10, fit_prior=False)
naive_model.fit(X, y)

new_loan = X.loc[146:146, :]
print('predicted class: ', naive_model.predict(new_loan)[0])

probabilities = pd.DataFrame(naive_model.predict_proba(new_loan),
                             columns=naive_model.classes_)
print('predicted probabilities',)
print(probabilities)

### Example not in book

**Koreksi catatan upstream:** GaussianNB mendukung fitur numerik. Contoh R berikut bukan sel Python wajib dan dipertahankan sebagai konteks. Implementasi Python numerik ada pada eksperimen tambahan.

Catatan asli mengenai keterbatasan implementasi tertentu tidak berlaku untuk seluruh keluarga Naive Bayes scikit-learn. The example would need to demonstrate binning a variable and display the probability distribution of the bins.
```
## example not in book
less_naive <- NaiveBayes(outcome ~ borrower_score + payment_inc_ratio + 
                           purpose_ + home_ + emp_len_, data = loan_data)
less_naive$table[1:2]

png(filename=file.path(PSDS_PATH, 'figures', 'psds_naive_bayes.png'),  width = 4, height=3, units='in', res=300)

stats <- less_naive$table[[1]]
ggplot(data.frame(borrower_score=c(0,1)), aes(borrower_score)) +
  stat_function(fun = dnorm, color='blue', linetype=1, 
                arg=list(mean=stats[1, 1], sd=stats[1, 2])) +
  stat_function(fun = dnorm, color='red', linetype=2, 
                arg=list(mean=stats[2, 1], sd=stats[2, 2])) +
  labs(y='probability')
dev.off()
```

# Discriminant Analysis
## A Simple Example

In [ ]:
loan3000 = pd.read_csv(LOAN3000_CSV)
loan3000.outcome = loan3000.outcome.astype('category')

predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

loan_lda = LinearDiscriminantAnalysis()
loan_lda.fit(X, y)
print(pd.DataFrame(loan_lda.scalings_, index=X.columns))

In [ ]:
pred = pd.DataFrame(loan_lda.predict_proba(loan3000[predictors]),
                    columns=loan_lda.classes_)
print(pred.head())

### Figure 5.1

In [ ]:
# Use scalings and center of means to determine decision boundary
center = np.mean(loan_lda.means_, axis=0)
slope = - loan_lda.scalings_[0] / loan_lda.scalings_[1]
intercept = center[1] - center[0] * slope

# payment_inc_ratio for borrower_score of 0 and 20
x_0 = (0 - intercept) / slope
x_20 = (20 - intercept) / slope

lda_df = pd.concat([loan3000, pred['default']], axis=1)
lda_df.head()

fig, ax = plt.subplots(figsize=(4, 4))
g = sns.scatterplot(x='borrower_score', y='payment_inc_ratio',
                    hue='default', data=lda_df, 
                    palette=sns.diverging_palette(240, 10, n=9, as_cmap=True),
                    ax=ax, legend=False)

ax.set_ylim(0, 20)
ax.set_xlim(0.15, 0.8)
ax.plot((x_0, x_20), (0, 20), linewidth=3)
ax.plot(*loan_lda.means_.transpose())

plt.tight_layout()
plt.show()

# Logistic regression
## Logistic Response Function and Logit

In [ ]:
p = np.arange(0.01, 1, 0.01)
df = pd.DataFrame({
    'p': p,
    'logit': np.log(p / (1 - p)),
    'odds': p / (1 - p),
})

fig, ax = plt.subplots(figsize=(3, 3))
ax.axhline(0, color='grey', linestyle='--')
ax.axvline(0.5, color='grey', linestyle='--')
ax.plot(df['p'], df['logit'])
ax.set_xlabel('Probability')
ax.set_ylabel('logit(p)')

plt.tight_layout()
plt.show()

## Logistic Regression and the GLM
The package _scikit-learn_ has a specialised class for `LogisticRegression`. _Statsmodels_ has a more general method based on generalized linear model (GLM).

In [ ]:
predictors = ['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_', 
              'borrower_score']
outcome = 'outcome'
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='', 
                   drop_first=True, dtype=int)
y = loan_data[outcome] # .cat.categories

logit_reg = LogisticRegression(C=1e42, solver='liblinear')
logit_reg.fit(X, y)

print('intercept ', logit_reg.intercept_[0])
print('classes', logit_reg.classes_)
pd.DataFrame({'coeff': logit_reg.coef_[0]}, 
             index=X.columns)

Note that the intercept and coefficients are reversed compared to the R model.

In [ ]:
print(loan_data['purpose_'].cat.categories)
print(loan_data['home_'].cat.categories)
print(loan_data['emp_len_'].cat.categories)

_Not in book_ :
If you have a feature or outcome variable that is ordinal, use the scikit-learn `OrdinalEncoder` to replace the categories (here, 'paid off' and 'default') with numbers. In the below code, we replace 'paid off' with 0 and 'default' with 1. This reverses the order of the predicted classes and as a consequence, the coefficients will be reversed. 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(categories=[['paid off', 'default']])
y_enc = enc.fit_transform(loan_data[[outcome]]).ravel()

logit_reg_enc = LogisticRegression(C=1e42, solver='liblinear')
logit_reg_enc.fit(X, y_enc)

print('intercept ', logit_reg_enc.intercept_[0])
print('classes', logit_reg_enc.classes_)
pd.DataFrame({'coeff': logit_reg_enc.coef_[0]}, 
             index=X.columns)

## Predicted Values from Logistic Regression

In [ ]:
pred = pd.DataFrame(logit_reg.predict_log_proba(X),
                    columns=logit_reg.classes_)
print(pred.describe())

In [ ]:
pred = pd.DataFrame(logit_reg.predict_proba(X),
                    columns=logit_reg.classes_)
print(pred.describe())

## Interpreting the Coefficients and Odds Ratios

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.plot(df['logit'], df['odds'])
ax.set_xlabel('log(odds ratio)')
ax.set_ylabel('odds ratio')
ax.set_xlim(0, 5.1)
ax.set_ylim(-5, 105)

plt.tight_layout()
plt.show()

## Assessing the Model
For comparison, here the GLM model using _statsmodels_. This method requires that the outcome is mapped to numbers. 

In [ ]:
# use GLM (general linear model) with the binomial family to 
# fit a logistic regression
y_numbers = [1 if yi == 'default' else 0 for yi in y]
logit_reg_sm = sm.GLM(y_numbers, X.assign(const=1), 
                      family=sm.families.Binomial())
logit_result = logit_reg_sm.fit()
print(logit_result.summary())

Use splines

In [ ]:
import statsmodels.formula.api as smf
formula = ('outcome ~ bs(payment_inc_ratio, df=8) + purpose_ + ' +
           'home_ + emp_len_ + bs(borrower_score, df=3)')
model = smf.glm(formula=formula, data=loan_data, family=sm.families.Binomial())
results = model.fit()
print(results.summary())

In [ ]:
from statsmodels.genmod.generalized_linear_model import GLMResults
def partialResidualPlot(model, df, outcome, feature, fig, ax):
    y_actual = [0 if s == 'default' else 1 for s in df[outcome]]
    y_pred = model.predict(df)
    org_params = model.params.copy()
    zero_params = model.params.copy()
    # set model parametes of other features to 0
    for i, name in enumerate(zero_params.index):
        if feature in name:
            continue
        zero_params.iloc[i] = 0.0
    model.initialize(model.model, zero_params)
    feature_prediction = model.predict(df)
    ypartial = -np.log(1/feature_prediction - 1)
    ypartial = ypartial - np.mean(ypartial)
    model.initialize(model.model, org_params)
    results = pd.DataFrame({
        'feature': df[feature],
        'residual': -2 * (y_actual - y_pred),
        'ypartial': ypartial/ 2,
    })
    results = results.sort_values(by=['feature'])

    ax.scatter(results.feature, results.residual, marker=".", s=72./fig.dpi)
    ax.plot(results.feature, results.ypartial, color='black')
    ax.set_xlabel(feature)
    ax.set_ylabel(f'Residual + {feature} contribution')
    return ax

fig, ax = plt.subplots(figsize=(5, 5))
partialResidualPlot(results, loan_data, 'outcome', 'payment_inc_ratio', fig, ax)
ax.set_xlim(0, 25)
ax.set_ylim(-2.5, 2.5)


plt.tight_layout()
plt.show()

# Evaluating Classification Models
## Confusion Matrix

In [ ]:
# Confusion matrix
pred = logit_reg.predict(X)
pred_y = logit_reg.predict(X) == 'default'
true_y = y == 'default'
true_pos = true_y & pred_y
true_neg = ~true_y & ~pred_y
false_pos = ~true_y & pred_y
false_neg = true_y & ~pred_y

conf_mat = pd.DataFrame([[np.sum(true_pos), np.sum(false_neg)], [np.sum(false_pos), np.sum(true_neg)]],
                       index=['Y = default', 'Y = paid off'],
                       columns=['Yhat = default', 'Yhat = paid off'])
print(conf_mat)

In [ ]:
print(confusion_matrix(y, logit_reg.predict(X)))

The package _dmba_ contains the function `classificationSummary` that prints confusion matrix and accuracy for a classification model. 

In [ ]:
classificationSummary(y, logit_reg.predict(X), 
                      class_names=logit_reg.classes_)

## Precision, Recall, and Specificity
The _scikit-learn_ function `precision_recall_fscore_support` returns
precision, recall, fbeta_score and support. 

In [ ]:
conf_mat = confusion_matrix(y, logit_reg.predict(X))
print('Precision', conf_mat[0, 0] / sum(conf_mat[:, 0]))
print('Recall', conf_mat[0, 0] / sum(conf_mat[0, :]))
print('Specificity', conf_mat[1, 1] / sum(conf_mat[1, :]))

In [ ]:
precision_recall_fscore_support(y, logit_reg.predict(X), 
                                labels=['default', 'paid off'])

## ROC Curve
The function `roc_curve` in _Scikit-learn_ calculates all the information that is required for plotting a ROC curve.

In [ ]:
fpr, tpr, thresholds = roc_curve(y, logit_reg.predict_proba(X)[:, 0], 
                                 pos_label='default')
roc_df = pd.DataFrame({'recall': tpr, 'specificity': 1 - fpr})

ax = roc_df.plot(x='specificity', y='recall', figsize=(4, 4), legend=False)
ax.set_ylim(0, 1)
ax.set_xlim(1, 0)
ax.plot((1, 0), (0, 1))
ax.set_xlabel('specificity')
ax.set_ylabel('recall')


plt.tight_layout()
plt.show()

## AUC
Accuracy can easily be calculated using the _scikit-learn_ function `accuracy_score`.

In [ ]:
print(np.sum(roc_df.recall[:-1] * np.diff(1 - roc_df.specificity)))
print(roc_auc_score([1 if yi == 'default' else 0 for yi in y], logit_reg.predict_proba(X)[:, 0]))

In [ ]:
fpr, tpr, thresholds = roc_curve(y, logit_reg.predict_proba(X)[:,0], 
                                 pos_label='default')
roc_df = pd.DataFrame({'recall': tpr, 'specificity': 1 - fpr})

ax = roc_df.plot(x='specificity', y='recall', figsize=(4, 4), legend=False)
ax.set_ylim(0, 1)
ax.set_xlim(1, 0)
# ax.plot((1, 0), (0, 1))
ax.set_xlabel('specificity')
ax.set_ylabel('recall')
ax.fill_between(roc_df.specificity, 0, roc_df.recall, alpha=0.3)


plt.tight_layout()
plt.show()

# Strategies for Imbalanced Data
## Undersampling
> The results differ from the R version, however are equivalent to results obtained using the R code. Model based results are of similar magnitude.

In [ ]:
full_train_set = pd.read_csv(FULL_TRAIN_SET_CSV)
print(full_train_set.shape)

In [ ]:
print('percentage of loans in default: ', 
print(      100 * np.mean(full_train_set.outcome == 'default')))

In [ ]:
predictors = ['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_', 
              'dti', 'revol_bal', 'revol_util']
outcome = 'outcome'
X = pd.get_dummies(full_train_set[predictors], prefix='', prefix_sep='', 
                   drop_first=True, dtype=int)
y = full_train_set[outcome]

full_model = LogisticRegression(C=1e42, solver='liblinear')
full_model.fit(X, y)
print('percentage of loans predicted to default: ', 
print(      100 * np.mean(full_model.predict(X) == 'default')))

In [ ]:
observed_default_rate = np.mean(full_train_set.outcome == 'default')
predicted_default_rate = np.mean(full_model.predict(X) == 'default')
if predicted_default_rate == 0:
    print('Rasio tidak terdefinisi: model tidak memprediksi satu pun default pada threshold bawaan.')
    print('Proporsi default aktual:', observed_default_rate)
else:
    print('Rasio proporsi aktual/prediksi:', observed_default_rate / predicted_default_rate)

## Oversampling and Up/Down Weighting

In [ ]:
default_wt = 1 / np.mean(full_train_set.outcome == 'default')
wt = [default_wt if outcome == 'default' else 1 for outcome in full_train_set.outcome]

full_model = LogisticRegression(C=1e42, solver='liblinear')
full_model.fit(X, y, wt)
print('percentage of loans predicted to default (weighting): ', 
print(      100 * np.mean(full_model.predict(X) == 'default')))

## Data Generation
The package _imbalanced-learn_ provides an implementation of the _SMOTE_ and similar algorithms. 

In [ ]:
X_resampled, y_resampled = SMOTE().fit_resample(X, y)
print('percentage of loans in default (SMOTE resampled): ', 
      100 * np.mean(y_resampled == 'default'))

full_model = LogisticRegression(C=1e42, solver='liblinear')
full_model.fit(X_resampled, y_resampled)
print('percentage of loans predicted to default (SMOTE): ', 
      100 * np.mean(full_model.predict(X) == 'default'))


X_resampled, y_resampled = ADASYN().fit_resample(X, y)
print('percentage of loans in default (ADASYN resampled): ', 
      100 * np.mean(y_resampled == 'default'))

full_model = LogisticRegression(C=1e42, solver='liblinear')
full_model.fit(X_resampled, y_resampled)
print('percentage of loans predicted to default (ADASYN): ', 
print(      100 * np.mean(full_model.predict(X) == 'default')))

## Exploring the Predictions

In [ ]:
loan3000 = pd.read_csv(LOAN3000_CSV)

predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

loan_tree = DecisionTreeClassifier(random_state=1, criterion='entropy', 
                                   min_impurity_decrease=0.003)
loan_tree.fit(X, y)

loan_lda = LinearDiscriminantAnalysis()
loan_lda.fit(X, y)

logit_reg = LogisticRegression(solver='liblinear')
logit_reg.fit(X, y)


## model
gam = LinearGAM(s(0) + s(1))
print(gam.gridsearch(X.values, [1 if yi == 'default' else 0 for yi in y]))

In [ ]:
models = {
    'Decision Tree': loan_tree,
    'Linear Discriminant Analysis': loan_lda,
    'Logistic Regression': logit_reg,
    'Generalized Additive Model': gam,
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(5, 5))

xvalues = np.arange(0.25, 0.73, 0.005)
yvalues = np.arange(-0.1, 20.1, 0.1)
xx, yy = np.meshgrid(xvalues, yvalues)
X = pd.DataFrame({
    'borrower_score': xx.ravel(),
    'payment_inc_ratio': yy.ravel(),
})

boundary = {}

for n, (title, model) in enumerate(models.items()):
    ax = axes[n // 2, n % 2]
    predict = model.predict(X)
    if 'Generalized' in title:
        Z = np.array([1 if z > 0.5 else 0 for z in predict])
    else:
        
        Z = np.array([1 if z == 'default' else 0 for z in predict])
    Z = Z.reshape(xx.shape)
    boundary[title] = yvalues[np.argmax(Z > 0, axis=0)]
    boundary[title][Z[-1,:] == 0] = yvalues[-1]

    c = ax.pcolormesh(xx, yy, Z, cmap='Blues', vmin=0.1, vmax=1.3, shading='auto')
    ax.set_title(title)
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
boundary['borrower_score'] = xvalues
boundaries = pd.DataFrame(boundary)

fig, ax = plt.subplots(figsize=(5, 4))
boundaries.plot(x='borrower_score', ax=ax)
ax.set_ylabel('payment_inc_ratio')
ax.set_ylim(0, 20)


plt.tight_layout()
plt.show()

# Eksperimen tambahan dan validasi
Sel berikut disusun sebagai pendalaman. Output muncul setelah sel kode dijalankan.

## Eksperimen tambahan: klasifikasi dengan holdout terstratifikasi
Enam fitur asli dipakai tanpa borrower_score. Dataset sumber sudah merupakan sampel tersusun; prevalensinya tidak diklaim mewakili populasi kredit. Semua scaler dipelajari di training melalui pipeline. Default=1.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score,average_precision_score,confusion_matrix
DATA=next(p/'data' for p in [Path.cwd(), *Path.cwd().parents] if (p/'data'/'state.csv').exists())
d=pd.read_csv(DATA/'loan_data.csv.gz')
features=['payment_inc_ratio','dti','revol_bal','revol_util','open_acc','annual_inc']
X=d[features]; y=(d.outcome=='default').astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
assert set(Xtr.index).isdisjoint(Xte.index)
print('Prevalensi default test:',yte.mean(),'; n train/test:',len(ytr),len(yte))

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
models={'Baseline prior':DummyClassifier(strategy='prior'),'Gaussian NB':make_pipeline(StandardScaler(),GaussianNB()),'LDA':make_pipeline(StandardScaler(),LinearDiscriminantAnalysis()),'Logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,random_state=42))}
rows=[]
for name,model in models.items():
    model.fit(Xtr,ytr); p=model.predict_proba(Xte)[:,list(model.classes_).index(1)]; pred=(p>=.5).astype(int)
    rows.append({'model':name,'accuracy':accuracy_score(yte,pred),'precision':precision_score(yte,pred,zero_division=0),'recall':recall_score(yte,pred),'ROC_AUC':roc_auc_score(yte,p),'average_precision':average_precision_score(yte,p)})
display(pd.DataFrame(rows))
p=models['Logistic'].predict_proba(Xte)[:,1]; pred=(p>=.5).astype(int)
display(pd.DataFrame(confusion_matrix(yte,pred,labels=[0,1]),index=['aktual paid off','aktual default'],columns=['prediksi paid off','prediksi default']))
top=np.argsort(-p)[:max(1,len(p)//10)]
print('Lift 10% teratas:',yte.iloc[top].mean()/yte.mean())

**Interpretasi:** baseline membatasi arti accuracy. ROC-AUC menilai ranking, sedangkan precision/recall bergantung threshold 0.5 yang ditetapkan di sini. Pemilihan threshold berdasarkan biaya harus dilakukan pada validation terpisah, bukan tabel test ini.

## Ringkasan hasil yang diperoleh

Jalankan semua sel sebelumnya, kemudian sel berikut untuk menampilkan ringkasan dari hasil perhitungan saat ini.

In [ ]:
# Metrik berasal dari model dan split pada eksperimen tambahan.
display(pd.DataFrame(rows))
print('Kelas positif: default (1). Interpretasikan metrik bersama prevalensi test.')